# ESLint Maintainability — Nesting Depth Raw Output Extraction (JavaScript)

This notebook analyzes **JavaScript repositories** with **ESLint** and captures the complete raw tool output for maintainability nesting-depth metric derivation and validation.

**Default benchmark repository:** [facebook/react](https://github.com/facebook/react)

The notebook supports:
- **Mode 1:** Clone from a Git repository URL
- **Mode 2:** Analyze an already-cloned local repository path

All deliverables are written to the configured `OUTPUT_DIR`.

**Prerequisite:** Node.js and npm must be available for ESLint execution.

## Section 1 — Install Dependencies

Install open-source Python packages and the ESLint CLI.

In [ ]:
!pip install -q pandas gitpython jupyter

In [ ]:
!npm install -g eslint

## Section 2 — Configuration

Set execution mode, repository source, output location, and ESLint nesting-depth rule options.

- Set `USE_GIT_URL = True` to clone from `REPO_URL`.
- Set `USE_GIT_URL = False` to analyze `LOCAL_REPO_PATH` directly.
- When cloning, use `IF_CLONE_EXISTS` to choose between reusing or re-cloning an existing local copy.

In [ ]:
USE_GIT_URL = True

REPO_URL = "https://github.com/facebook/react.git"

LOCAL_REPO_PATH = "/content/react"

OUTPUT_DIR = "./outputs"

ESLINT_RULE = "max-depth"

MAX_ALLOWED_DEPTH = 5

IF_CLONE_EXISTS = "reuse"

CLONE_DEPTH = 1

WORKSPACE_DIR = "./workspace"

STREAM_RAW_OUTPUT = True

RAW_OUTPUT_PREVIEW_LINES = 150

# Fast validation benchmark (100% predictable nesting outcomes):
# USE_GIT_URL = False
# LOCAL_REPO_PATH = "./workspace/js_nesting_benchmark"

## Utility Functions

Modular helpers for logging, repository setup, JavaScript file discovery, ESLint execution, and nesting-depth extraction.

In [ ]:
from __future__ import annotations

import json
import os
import re
import shutil
import subprocess
import sys
import threading
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

os.environ.pop("PYTHONPATH", None)
sys.path = [
    path for path in sys.path
    if "site-packages" in path.replace("\\", "/") or path in {"", "."}
]

import pandas as pd
from IPython.display import display
from git import Repo
from git.exc import GitCommandError, InvalidGitRepositoryError


JS_EXTENSIONS = {".js", ".jsx", ".mjs", ".cjs"}

EXCLUDED_DIR_NAMES = {
    ".git",
    "node_modules",
    "dist",
    "build",
    "coverage",
    "out",
    "vendor",
    "docs",
}

MAX_DEPTH_MESSAGE_PATTERN = re.compile(
    r"Blocks are nested too deeply \((?P<actual>\d+)\)\. Maximum allowed is (?P<threshold>\d+)\."
)


class NotebookLogger:
    def __init__(self, error_log_path: Path) -> None:
        self.error_log_path = error_log_path
        self.error_log_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.error_log_path.exists():
            self.error_log_path.write_text("", encoding="utf-8")

    def info(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        print(f"[{timestamp}] INFO: {message}")

    def error(self, message: str) -> None:
        timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        line = f"[{timestamp}] ERROR: {message}\n"
        print(line, end="")
        with self.error_log_path.open("a", encoding="utf-8") as handle:
            handle.write(line)


def ensure_output_dir(output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)


def derive_clone_path(repo_url: str, workspace_dir: Path) -> Path:
    repo_name = repo_url.rstrip("/").removesuffix(".git").split("/")[-1]
    if not repo_name:
        raise ValueError(f"Unable to derive repository name from URL: {repo_url}")
    return workspace_dir / repo_name


def validate_repo_url(repo_url: str) -> None:
    if not repo_url or not isinstance(repo_url, str):
        raise ValueError("REPO_URL must be a non-empty string.")
    if not (repo_url.startswith("https://") or repo_url.startswith("git@") or repo_url.startswith("http://")):
        raise ValueError(f"Invalid repository URL format: {repo_url}")


def clone_or_reuse_repository(
    repo_url: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    validate_repo_url(repo_url)
    workspace_dir.mkdir(parents=True, exist_ok=True)
    clone_path = derive_clone_path(repo_url, workspace_dir)

    if clone_path.exists():
        if if_clone_exists == "reclone":
            logger.info(f"Removing existing clone at {clone_path}")
            shutil.rmtree(clone_path)
        elif if_clone_exists == "reuse":
            logger.info(f"Reusing existing clone at {clone_path}")
            return clone_path.resolve()
        else:
            raise ValueError('IF_CLONE_EXISTS must be either "reuse" or "reclone"')

    logger.info(f"Cloning {repo_url} into {clone_path} (depth={clone_depth})")
    try:
        clone_kwargs = {"depth": clone_depth} if clone_depth else {}
        Repo.clone_from(repo_url, clone_path, **clone_kwargs)
    except GitCommandError as exc:
        logger.error(f"Clone failed for {repo_url}: {exc}")
        raise

    logger.info(f"Clone completed: {clone_path}")
    return clone_path.resolve()


def validate_local_repository(local_repo_path: Path, logger: NotebookLogger) -> Path:
    if not local_repo_path.exists():
        message = f"Local repository path does not exist: {local_repo_path}"
        logger.error(message)
        raise FileNotFoundError(message)

    if not local_repo_path.is_dir():
        message = f"Local repository path is not a directory: {local_repo_path}"
        logger.error(message)
        raise NotADirectoryError(message)

    has_git = (local_repo_path / ".git").exists()
    has_js = any(
        file_path.suffix.lower() in JS_EXTENSIONS
        for file_path in local_repo_path.rglob("*")
        if file_path.is_file()
    )

    if not has_git and not has_js:
        message = (
            f"Path is neither a Git repository nor a JavaScript source directory: {local_repo_path}"
        )
        logger.error(message)
        raise ValueError(message)

    if has_git:
        try:
            Repo(local_repo_path)
        except InvalidGitRepositoryError as exc:
            logger.error(f"Invalid Git repository at {local_repo_path}: {exc}")
            raise

    logger.info(f"Validated local repository path: {local_repo_path}")
    return local_repo_path.resolve()


def resolve_repository_path(
    use_git_url: bool,
    repo_url: str,
    local_repo_path: str,
    workspace_dir: Path,
    if_clone_exists: str,
    logger: NotebookLogger,
    clone_depth: int | None = None,
) -> Path:
    if use_git_url:
        logger.info("Execution mode: Git repository URL")
        return clone_or_reuse_repository(
            repo_url, workspace_dir, if_clone_exists, logger, clone_depth=clone_depth
        )
    logger.info("Execution mode: Local repository path")
    return validate_local_repository(Path(local_repo_path), logger)


def should_exclude_path(path: Path) -> bool:
    return any(part in EXCLUDED_DIR_NAMES for part in path.parts)


def discover_javascript_files(repo_path: Path) -> list[Path]:
    js_files: list[Path] = []
    for file_path in repo_path.rglob("*"):
        if not file_path.is_file():
            continue
        if file_path.suffix.lower() not in JS_EXTENSIONS:
            continue
        if should_exclude_path(file_path.relative_to(repo_path)):
            continue
        js_files.append(file_path.resolve())
    js_files.sort()
    return js_files


def compute_repository_stats(repo_path: Path, js_files: list[Path]) -> dict[str, Any]:
    total_size_bytes = sum(file_path.stat().st_size for file_path in js_files)
    directory_count = sum(
        1
        for current_path, _, _ in os.walk(repo_path)
        if not should_exclude_path(Path(current_path).relative_to(repo_path))
    )
    return {
        "javascript_file_count": len(js_files),
        "repository_size_bytes": total_size_bytes,
        "directory_count": directory_count,
    }


def save_javascript_file_list(js_files: list[Path], repo_path: Path, output_csv: Path) -> None:
    rows = [
        {
            "absolute_path": str(file_path),
            "relative_path": str(file_path.relative_to(repo_path)),
            "extension": file_path.suffix.lower(),
        }
        for file_path in js_files
    ]
    pd.DataFrame(rows).to_csv(output_csv, index=False)


def verify_node_runtime(logger: NotebookLogger) -> None:
    for command_name, args in (("node", ["-v"]), ("npm", ["-v"])):
        executable = shutil.which(command_name)
        if not executable:
            logger.error(f"Required command not found: {command_name}")
            raise RuntimeError("Node.js and npm are required for ESLint execution.")
        command = [executable, *args]
        try:
            completed = subprocess.run(
                command,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                check=False,
            )
        except FileNotFoundError as exc:
            logger.error(f"Required command not found: {command_name}")
            raise RuntimeError("Node.js and npm are required for ESLint execution.") from exc
        if completed.returncode != 0:
            logger.error(f"Command failed: {' '.join(command)} -> {completed.stderr}")
            raise RuntimeError(f"Unable to verify {command_name} runtime.")


def resolve_eslint_executable(project_root: Path | None = None) -> list[str]:
    search_roots: list[Path] = []
    if project_root is not None:
        search_roots.append(project_root)
    search_roots.append(Path.cwd())

    for base in search_roots:
        local = base / "node_modules" / ".bin" / "eslint"
        for candidate in (local.with_suffix(".cmd"), local):
            if candidate.exists():
                return [str(candidate.resolve())]

    eslint_path = shutil.which("eslint")
    if eslint_path:
        return [eslint_path]

    npx_path = shutil.which("npx")
    if npx_path:
        return [npx_path, "eslint"]

    raise FileNotFoundError("ESLint not found. Install with: npm install -g eslint")


def ensure_eslint_config(
    repo_path: Path,
    max_allowed_depth: int,
    eslint_rule: str,
    logger: NotebookLogger,
) -> tuple[Path, Path]:
    eslintrc_path = repo_path / ".eslintrc.json"
    flat_config_path = repo_path / "eslint.config.js"

    eslintrc_content = {
        "env": {"browser": True, "node": True, "es2022": True},
        "parserOptions": {"ecmaVersion": "latest", "sourceType": "module"},
        "rules": {"max-depth": ["warn", max_allowed_depth]},
    }

    if not eslintrc_path.exists():
        eslintrc_path.write_text(json.dumps(eslintrc_content, indent=2), encoding="utf-8")
        logger.info(f"Created ESLint legacy config: {eslintrc_path}")

    if not flat_config_path.exists():
        flat_config = (
            "export default [\n"
            "  {\n"
            "    files: ['**/*.{js,jsx,mjs,cjs}'],\n"
            "    languageOptions: { ecmaVersion: 'latest', sourceType: 'module' },\n"
            f"    rules: {{ '{eslint_rule}': ['warn', {max_allowed_depth}] }},\n"
            "  },\n"
            "];\n"
        )
        flat_config_path.write_text(flat_config, encoding="utf-8")
        logger.info(f"Created ESLint flat config: {flat_config_path}")

    return eslintrc_path, flat_config_path


def build_eslint_command(
    repo_path: Path,
    max_allowed_depth: int,
    output_format: str | None = None,
    project_root: Path | None = None,
) -> list[str]:
    command = resolve_eslint_executable(project_root=project_root) + [str(repo_path)]
    command.extend(["--rule", f"max-depth:[\"warn\",{max_allowed_depth}]"])
    if output_format:
        command.extend(["-f", output_format])
    return command


def run_eslint_command(
    command: list[str],
    logger: NotebookLogger,
    stream_raw: bool = False,
) -> tuple[str, str, int, bool]:
    try:
        if stream_raw:
            process = subprocess.Popen(
                command,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                encoding="utf-8",
                errors="replace",
                env=os.environ.copy(),
            )
            stdout_lines: list[str] = []
            stderr_lines: list[str] = []

            def _read_stream(pipe, sink, label):
                for line in iter(pipe.readline, ""):
                    print(line, end="", file=sys.stderr if label == "stderr" else sys.stdout)
                    sink.append(line)
                pipe.close()

            assert process.stdout is not None
            assert process.stderr is not None
            stdout_thread = threading.Thread(
                target=_read_stream, args=(process.stdout, stdout_lines, "stdout"), daemon=True
            )
            stderr_thread = threading.Thread(
                target=_read_stream, args=(process.stderr, stderr_lines, "stderr"), daemon=True
            )
            stdout_thread.start()
            stderr_thread.start()
            return_code = process.wait()
            stdout_thread.join()
            stderr_thread.join()
            stdout = "".join(stdout_lines)
            stderr = "".join(stderr_lines)
        else:
            completed = subprocess.run(
                command,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
                check=False,
                env=os.environ.copy(),
            )
            stdout = completed.stdout
            stderr = completed.stderr
            return_code = completed.returncode

        success = return_code in (0, 1)
        if not success:
            logger.error(
                f"ESLint command failed with exit code {return_code}: {' '.join(command)}"
            )
        return stdout, stderr, return_code, success
    except Exception as exc:
        logger.error(f"ESLint execution exception: {exc}")
        return "", str(exc), -1, False


def combine_raw_streams(stdout: str, stderr: str) -> str:
    raw_output = stdout
    if stderr:
        if raw_output and not raw_output.endswith("\n"):
            raw_output += "\n"
        raw_output += stderr
    return raw_output


def run_eslint_suite(
    repo_path: Path,
    max_allowed_depth: int,
    logger: NotebookLogger,
    stream_raw: bool = False,
    project_root: Path | None = None,
) -> dict[str, str]:
    verify_node_runtime(logger)
    logger.info(f"Running ESLint on repository: {repo_path}")

    text_command = build_eslint_command(
        repo_path, max_allowed_depth, project_root=project_root
    )
    text_stdout, text_stderr, _, text_ok = run_eslint_command(
        text_command, logger, stream_raw=stream_raw
    )

    json_command = build_eslint_command(
        repo_path, max_allowed_depth, output_format="json", project_root=project_root
    )
    json_stdout, json_stderr, _, json_ok = run_eslint_command(json_command, logger, stream_raw=False)
    if json_stderr.strip():
        logger.error(f"ESLint JSON stderr: {json_stderr.strip()}")

    return {
        "raw_text": combine_raw_streams(text_stdout, text_stderr),
        "json_text": json_stdout,
        "execution_ok": str(text_ok or json_ok),
    }


def parse_eslint_json(json_text: str, logger: NotebookLogger) -> list[dict[str, Any]]:
    if not json_text.strip():
        return []
    try:
        parsed = json.loads(json_text)
    except json.JSONDecodeError as exc:
        logger.error(f"Failed to parse ESLint JSON output: {exc}")
        return []
    if isinstance(parsed, list):
        return parsed
    logger.error("Unexpected ESLint JSON payload; expected a list.")
    return []


def eslint_json_to_dataframe(records: list[dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for record in records:
        file_path = record.get("filePath", "")
        for message in record.get("messages", []):
            rows.append(
                {
                    "file": file_path,
                    "line": message.get("line", ""),
                    "column": message.get("column", ""),
                    "severity": message.get("severity", ""),
                    "ruleId": message.get("ruleId", ""),
                    "message": message.get("message", ""),
                    "nodeType": message.get("nodeType", ""),
                }
            )
    columns = ["file", "line", "column", "severity", "ruleId", "message", "nodeType"]
    return pd.DataFrame(rows, columns=columns)


def count_eslint_failures(records: list[dict[str, Any]]) -> int:
    return sum(1 for record in records if record.get("fatalErrorCount", 0) > 0)


def extract_depth_from_message(message: str, default_threshold: int) -> tuple[int | None, int]:
    match = MAX_DEPTH_MESSAGE_PATTERN.search(message)
    if match:
        return int(match.group("actual")), int(match.group("threshold"))
    return None, default_threshold


def build_nesting_depth_findings(
    results_df: pd.DataFrame,
    eslint_rule: str,
    default_threshold: int,
) -> pd.DataFrame:
    if results_df.empty:
        return pd.DataFrame(
            columns=["file", "line", "rule", "actual_depth", "threshold", "status"]
        )

    nesting_df = results_df[results_df["ruleId"] == eslint_rule].copy()
    rows = []
    for _, record in nesting_df.iterrows():
        message = str(record.get("message", ""))
        actual_depth, threshold = extract_depth_from_message(message, default_threshold)
        rows.append(
            {
                "file": record.get("file", ""),
                "line": record.get("line", ""),
                "rule": record.get("ruleId", ""),
                "actual_depth": actual_depth,
                "threshold": threshold,
                "status": "reported" if actual_depth is not None else "unparsed",
            }
        )
    return pd.DataFrame(rows)


def compute_maintainability_summary(findings_df: pd.DataFrame) -> pd.DataFrame:
    valid = findings_df[findings_df["actual_depth"].notna()].copy()
    if valid.empty:
        return pd.DataFrame(
            [
                {"metric_name": "Maintainability_Nesting_Depth", "metric_value": 0},
                {"metric_name": "Average_Nesting_Depth", "metric_value": 0},
            ]
        )
    max_depth = int(valid["actual_depth"].max())
    avg_depth = float(valid["actual_depth"].mean())
    return pd.DataFrame(
        [
            {"metric_name": "Maintainability_Nesting_Depth", "metric_value": max_depth},
            {"metric_name": "Average_Nesting_Depth", "metric_value": round(avg_depth, 4)},
        ]
    )


def preview_raw_output(raw_text: str, preview_lines: int, output_path: Path) -> None:
    lines = raw_text.splitlines()
    print(f"\n{'=' * 80}")
    print(f"RAW ESLINT OUTPUT PREVIEW (first {preview_lines} lines)")
    print(f"{'=' * 80}\n")
    if not lines:
        print("(empty raw output)")
        return
    print("\n".join(lines[:preview_lines]))
    remaining = len(lines) - preview_lines
    if remaining > 0:
        print(f"\n... ({remaining} more lines saved to {output_path})")

## Section 3 — Repository Setup

Resolve the repository path based on configuration and initialize output directories.

In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
ERROR_LOG_PATH = OUTPUT_PATH / "error_log.txt"

ensure_output_dir(OUTPUT_PATH)
logger = NotebookLogger(ERROR_LOG_PATH)

try:
    REPO_PATH = resolve_repository_path(
        use_git_url=USE_GIT_URL,
        repo_url=REPO_URL,
        local_repo_path=LOCAL_REPO_PATH,
        workspace_dir=WORKSPACE_PATH,
        if_clone_exists=IF_CLONE_EXISTS,
        logger=logger,
        clone_depth=CLONE_DEPTH,
    )
except Exception as exc:
    logger.error(f"Repository setup failed: {exc}")
    raise

logger.info(f"Repository ready at: {REPO_PATH}")

## Section 4 — Discover JavaScript Files

Recursively discover `.js`, `.jsx`, `.mjs`, and `.cjs` files while excluding build and dependency directories.

In [ ]:
JS_FILES = discover_javascript_files(REPO_PATH)
REPO_STATS = compute_repository_stats(REPO_PATH, JS_FILES)

JAVASCRIPT_FILES_CSV = OUTPUT_PATH / "javascript_files.csv"
save_javascript_file_list(JS_FILES, REPO_PATH, JAVASCRIPT_FILES_CSV)

print(f"Total JavaScript Files Found: {len(JS_FILES)}")
print(f"Repository Size (JavaScript files only): {REPO_STATS['repository_size_bytes']:,} bytes")
print(f"Total Directories (excluding filtered paths): {REPO_STATS['directory_count']:,}")
print(f"Saved file list to: {JAVASCRIPT_FILES_CSV}")

## Section 5 — Create ESLint Configuration

Create `.eslintrc.json` (legacy format) and `eslint.config.js` (ESLint 9+ flat config) when missing.

Both configurations enable the `max-depth` rule with the configured threshold.

In [ ]:
ESLINTRC_PATH, FLAT_CONFIG_PATH = ensure_eslint_config(
    repo_path=REPO_PATH,
    max_allowed_depth=MAX_ALLOWED_DEPTH,
    eslint_rule=ESLINT_RULE,
    logger=logger,
)

print(f"ESLint legacy config: {ESLINTRC_PATH}")
print(f"ESLint flat config: {FLAT_CONFIG_PATH}")
print(f"Rule: {ESLINT_RULE} with threshold {MAX_ALLOWED_DEPTH}")

## Section 6 — Execute ESLint

Run ESLint against the repository and capture complete raw output.

Example equivalent command:

```bash
eslint <repo_path> --rule 'max-depth:["warn",5]'
```

Raw stdout and stderr are preserved without suppression.

In [ ]:
if not JS_FILES:
    logger.error("No JavaScript files discovered; skipping ESLint execution.")
    ESLINT_OUTPUTS = {"raw_text": "", "json_text": "", "execution_ok": "False"}
    FILES_SUCCESS = 0
    FILES_FAILED = 0
else:
    ESLINT_OUTPUTS = run_eslint_suite(
        repo_path=REPO_PATH,
        max_allowed_depth=MAX_ALLOWED_DEPTH,
        logger=logger,
        stream_raw=STREAM_RAW_OUTPUT,
        project_root=(Path.cwd() / "../../runtimes").resolve(),
    )
    ESLINT_JSON_RECORDS = parse_eslint_json(ESLINT_OUTPUTS["json_text"], logger)
    FILES_FAILED = count_eslint_failures(ESLINT_JSON_RECORDS)
    FILES_SUCCESS = len(JS_FILES) - FILES_FAILED

logger.info(f"ESLint execution complete. Raw output size: {len(ESLINT_OUTPUTS['raw_text']):,} characters")

## Section 7 — Raw Output Extraction

Persist complete raw ESLint text output, JSON output, and a CSV representation of all findings.

In [ ]:
RAW_OUTPUT_PATH = OUTPUT_PATH / "eslint_raw_output.txt"
JSON_OUTPUT_PATH = OUTPUT_PATH / "eslint_output.json"
RESULTS_CSV_PATH = OUTPUT_PATH / "eslint_results.csv"

raw_text_output = ESLINT_OUTPUTS["raw_text"]
RAW_OUTPUT_PATH.write_text(raw_text_output, encoding="utf-8")

if "ESLINT_JSON_RECORDS" not in globals():
    ESLINT_JSON_RECORDS = parse_eslint_json(ESLINT_OUTPUTS["json_text"], logger)

JSON_OUTPUT_PATH.write_text(
    json.dumps(ESLINT_JSON_RECORDS, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

ESLINT_RESULTS_DF = eslint_json_to_dataframe(ESLINT_JSON_RECORDS)
ESLINT_RESULTS_DF.to_csv(RESULTS_CSV_PATH, index=False)

logger.info(f"Saved raw output: {RAW_OUTPUT_PATH}")
logger.info(f"Saved JSON output: {JSON_OUTPUT_PATH}")
logger.info(f"Saved CSV results: {RESULTS_CSV_PATH}")
logger.info(f"Total ESLint findings: {len(ESLINT_RESULTS_DF)}")

preview_raw_output(raw_text_output, RAW_OUTPUT_PREVIEW_LINES, RAW_OUTPUT_PATH)

## Section 8 — Maintainability Nesting Depth Extraction

Extract all violations produced by the `max-depth` rule.

Parse ESLint messages such as:

```text
Blocks are nested too deeply (6). Maximum allowed is 5.
```

In [ ]:
NESTING_FINDINGS_DF = build_nesting_depth_findings(
    results_df=ESLINT_RESULTS_DF,
    eslint_rule=ESLINT_RULE,
    default_threshold=MAX_ALLOWED_DEPTH,
)

NESTING_FINDINGS_CSV = OUTPUT_PATH / "nesting_depth_findings.csv"
NESTING_FINDINGS_DF.to_csv(NESTING_FINDINGS_CSV, index=False)

logger.info(f"Saved nesting depth findings: {NESTING_FINDINGS_CSV}")
logger.info(f"Nesting depth findings count: {len(NESTING_FINDINGS_DF)}")

if not NESTING_FINDINGS_DF.empty:
    display(NESTING_FINDINGS_DF.head(10))
else:
    print("No nesting depth findings detected.")

## Section 9 — Metric Computation

Compute repository-level maintainability nesting depth metrics:

- **Maintainability_Nesting_Depth** = `max(actual_depth)`
- **Average_Nesting_Depth** = `Σ actual_depth / total_violating_functions`

In [ ]:
SUMMARY_DF = compute_maintainability_summary(NESTING_FINDINGS_DF)
SUMMARY_CSV = OUTPUT_PATH / "maintainability_nesting_depth_summary.csv"
SUMMARY_DF.to_csv(SUMMARY_CSV, index=False)

logger.info(f"Saved maintainability summary: {SUMMARY_CSV}")
display(SUMMARY_DF)

## Section 10 — Summary Dashboard

Overview of analysis coverage, ESLint findings, and nesting-depth metrics.

In [ ]:
max_depth = SUMMARY_DF.loc[
    SUMMARY_DF["metric_name"] == "Maintainability_Nesting_Depth", "metric_value"
].iloc[0]
avg_depth = SUMMARY_DF.loc[
    SUMMARY_DF["metric_name"] == "Average_Nesting_Depth", "metric_value"
].iloc[0]

summary_df = pd.DataFrame(
    [
        {"Metric": "Total JavaScript Files", "Value": len(JS_FILES)},
        {"Metric": "Files Successfully Analyzed", "Value": FILES_SUCCESS},
        {"Metric": "Files Failed", "Value": FILES_FAILED},
        {"Metric": "Total ESLint Findings", "Value": len(ESLINT_RESULTS_DF)},
        {"Metric": "Nesting Depth Findings", "Value": len(NESTING_FINDINGS_DF)},
        {"Metric": "Maximum Nesting Depth", "Value": max_depth},
        {"Metric": "Average Nesting Depth", "Value": avg_depth},
    ]
)

display(summary_df)

deliverables = [
    RAW_OUTPUT_PATH,
    JSON_OUTPUT_PATH,
    RESULTS_CSV_PATH,
    JAVASCRIPT_FILES_CSV,
    NESTING_FINDINGS_CSV,
    SUMMARY_CSV,
    ERROR_LOG_PATH,
]

print("\nDeliverables:")
for deliverable in deliverables:
    status = "OK" if deliverable.exists() else "MISSING"
    print(f"  [{status}] {deliverable}")

## Section 11 — Error Handling

Failures encountered during cloning, validation, or ESLint execution are appended to `outputs/error_log.txt`.

In [ ]:
if ERROR_LOG_PATH.exists() and ERROR_LOG_PATH.stat().st_size > 0:
    print(ERROR_LOG_PATH.read_text(encoding="utf-8"))
else:
    print("No errors logged.")

## Section 12 — Deliverables

Upon successful completion, the following artifacts are available under `outputs/`:

```text
outputs/
├── eslint_raw_output.txt
├── eslint_output.json
├── eslint_results.csv
├── javascript_files.csv
├── nesting_depth_findings.csv
├── maintainability_nesting_depth_summary.csv
└── error_log.txt
```

The notebook is designed to run end-to-end in Jupyter Notebook and Google Colab without manual intervention.